# Lesson 8 (part 1): KKBox data preparation

**Why this lesson?** In Lesson 7, deep learning didn't beat XGBoost on the Telco data. Telco has **one snapshot row per customer**, and deep learning's real strength is learning from **raw sequences**.
So in Lesson 8 we switch to **KKBox**, a music streaming service, and use its **daily listening logs**: each user becomes a **sequence of 21 days**.

Lesson 8 has two notebooks:
- **06 (this one), data preparation:** extract the big file, sample users, clean, explore, build the labels and the sequences.
- **07, sequence models:** XGBoost baselines, LSTM and GRU, and a fair comparison.

**This notebook, step by step:**
1. Extract the archive and read the big file in chunks.
2. Keep a sample of **users** (not rows), so every kept user's history is complete.
3. Clean the data and look at it.
4. Define the churn label **without data leakage**, using a time split.
5. Turn each user into a 21-day sequence, and save it for notebook 07.

## The data
We use one file from the Kaggle competition "WSDM - KKBox's Churn Prediction Challenge": `user_logs_v2.csv`, the **daily listening logs for March 2017** (about 18 million rows, over 1 GB).

| Column | Meaning |
|---|---|
| `msno` | user id |
| `date` | the day, written as YYYYMMDD (for example 20170315) |
| `num_25` | songs played for less than 25% of their length (skipped quickly) |
| `num_50` | songs played for 25% to 50% of their length |
| `num_75` | songs played for 50% to 75% of their length |
| `num_985` | songs played for 75% to 98.5% of their length |
| `num_100` | songs played for more than 98.5% of their length (listened to the end) |
| `num_unq` | number of different songs played |
| `total_secs` | total seconds listened that day |

**Important:** a row exists **only on days the user listened**. A missing day means no listening.

## Why we define our own churn label
This is an important design decision.

**The official KKBox label** means "did not renew the paid subscription within 30 days after it expired", for subscriptions expiring during March 2017.
Our log file covers March itself, so for many users it includes days **after** their subscription ended. Using it with the official labels would let the model **peek at the outcome**: that is **data leakage**.
Doing it properly would need the full log history (about 30 GB), which is too heavy for a laptop.

So we define **engagement churn** instead. It is very common in industry for apps ("no activity for N days"), and we use a clean **time split**:

```
March 2017
Days 1 to 21     OBSERVATION WINDOW   the model's input: 21 days of daily listening behavior
Days 22 to 31    PREDICTION WINDOW    the label: churn = 1 if the user does NOT listen at all here
```

- **Who is included:** only users who listened at least once in **days 15 to 21** ("currently active users"). Users who had already stopped would be trivially easy to predict, and the business wants to catch **active** users who are about to leave.
- The input window and the label window **never overlap**, so the model never sees the future: no leakage.

# Step 1: Extract and read the data

**Imports and file paths.**
- `os`, `glob`, `shutil` and `subprocess` work with files and folders: check whether a file exists, search for a file, move it, and run a command-line tool.
- `time` measures how long the slow steps take.
- pandas, NumPy and matplotlib as always.
- `DATA_DIR` is the folder with the KKBox files, `ARCHIVE_PATH` the downloaded `.7z` file, and `LOGS_PATH` the extracted CSV.
- The folder `data/kkbox/` is listed in `.gitignore`, so these big files are never added to git (GitHub refuses files over 100 MB).

In [ ]:
import os
import glob
import shutil
import subprocess
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "../data/kkbox"
ARCHIVE_PATH = f"{DATA_DIR}/user_logs_v2.csv.7z"
LOGS_PATH = f"{DATA_DIR}/user_logs_v2.csv"

**Extract the archive (only if needed).**
- A `.7z` file is a **compressed archive**, like a `.zip`. We extract it only once: if the CSV already exists, this cell just says so.
- If **neither** the CSV nor the archive is there, the cell stops with a clear message (`raise FileNotFoundError`) saying which file to download.
- **Two ways to extract:**
  - normally with **py7zr**, a Python library for `.7z` files;
  - if py7zr can't run, with the **7-Zip command-line tool** (`7zz`, installed with `brew install sevenzip`). This is the case on this Mac: its Python was built without LZMA support, which py7zr needs.

  `try` / `except ImportError` means: try py7zr, and if importing it fails, use 7-Zip instead. (`x` means "extract with folders", `-o` sets the output folder and `-y` answers "yes" to any question.)
- The CSV inside the archive may sit in a **subfolder**, so we search for it (`glob` with `**` searches all subfolders) and move it to a fixed path.
- At the end we print how long it took and the file size.

In [ ]:
if os.path.exists(LOGS_PATH):
    print("CSV already extracted:", LOGS_PATH)
elif not os.path.exists(ARCHIVE_PATH):
    raise FileNotFoundError("Download user_logs_v2.csv.7z from the Kaggle competition "
                            "'WSDM - KKBox's Churn Prediction Challenge' into data/kkbox/ first")
else:
    start = time.time()
    try:
        import py7zr
        with py7zr.SevenZipFile(ARCHIVE_PATH, mode="r") as archive:
            archive.extractall(path=DATA_DIR)
    except ImportError:
        print("py7zr can't run on this Python, so we use the 7-Zip command-line tool instead")
        subprocess.run(["7zz", "x", ARCHIVE_PATH, f"-o{DATA_DIR}", "-y"], check=True, capture_output=True)
    found = glob.glob(f"{DATA_DIR}/**/user_logs_v2.csv", recursive=True)
    print("Found:", found)
    if found and found[0] != LOGS_PATH:
        shutil.move(found[0], LOGS_PATH)
    print(f"Extracted in {time.time() - start:.0f} seconds")

print("File size (GB):", round(os.path.getsize(LOGS_PATH) / 1e9, 2))

**Read the big file in chunks, and keep a sample of users.**
- **Chunks:** `chunksize=2_000_000` reads 2 million rows at a time. Each chunk is filtered before the next one is read, so memory stays low, instead of loading about 18 million rows at once.
- **Sample users, not rows.** If we randomly kept rows, every user would lose random days, and their sequences would be broken. Instead we keep **all the days of about 1 in 10 users**.
- **How users are chosen:** `pd.util.hash_pandas_object(chunk["msno"], index=False)` turns each user id into a big number with a **fixed rule** (the same id always gives the same number). `% SAMPLE_EVERY == 0` keeps the users whose number is divisible by 10, i.e. ends in 0.
  So the **same users are chosen in every chunk and in every run** (reproducible), and a user's rows are never split between "kept" and "dropped".
- `SAMPLE_EVERY = 10` keeps about 10% of the users. On a slow laptop, 20 keeps about 5%.
- `usecols=cols` reads only the columns we need.

In [ ]:
SAMPLE_EVERY = 10   # keep about 1 in 10 users (set to 20 if your laptop is slow)

cols = ["msno", "date", "num_25", "num_50", "num_75", "num_985", "num_100", "num_unq", "total_secs"]
parts = []
total_rows = 0
start = time.time()

for chunk in pd.read_csv(LOGS_PATH, usecols=cols, chunksize=2_000_000):
    total_rows += len(chunk)
    keep = pd.util.hash_pandas_object(chunk["msno"], index=False) % SAMPLE_EVERY == 0
    parts.append(chunk[keep])

logs = pd.concat(parts, ignore_index=True)
print(f"Read {total_rows:,} rows in {time.time() - start:.0f} seconds")
print(f"Kept {len(logs):,} rows for {logs['msno'].nunique():,} users")

# Step 2: Check and clean the data

**Data quality check.** `describe()` shows, for every numeric column, the count, mean, standard deviation, minimum, quartiles and maximum. `.T` flips the table, so each column becomes a row.

What to look for:
- **negative values**: impossible for counts of songs or seconds;
- **absurd `total_secs`**: a day has only 86,400 seconds (24 hours);
- **dates** outside March 2017.

Real logs are messy, so we always check before modelling.

In [ ]:
logs.describe().T

**Count the problems before fixing them.** Before changing anything, we count how many rows are affected by each problem, so we know how big each fix is.
`(logs[num_cols] < 0).sum()` counts the negative values in each column (True counts as 1).

In [ ]:
num_cols = ["num_25", "num_50", "num_75", "num_985", "num_100", "num_unq", "total_secs"]

print("Missing values:", logs[num_cols].isnull().sum().sum())
print()
print("Negative values in each column:")
print((logs[num_cols] < 0).sum())
print()
print("Rows with more than 24 hours of listening:", (logs["total_secs"] > 86400).sum())

*Pending: this explanation is written from the real results, after the first run on the KKBox data.* <!-- PLACEHOLDER_QUALITY -->

**Clean the data.** Each rule, and why:
1. **Remove exact duplicate rows** (`drop_duplicates()`): the same user, day and numbers twice is a logging error, so we keep one copy.
2. **Remove rows with negative or missing values:** a negative number of songs or seconds is impossible. `logs[num_cols] >= 0` is True only for valid values (it is also False for missing values), and `.all(axis=1)` keeps a row only if **all** its values are valid.
3. **Clip `total_secs` to 24 hours:** `clip(upper=86400)` caps values above 86,400 seconds and leaves the others unchanged. We keep these rows, because the user clearly did listen that day; only the seconds are wrong.
4. **Keep March 2017 only.**
5. **Day of the month:** `date % 100` is the remainder after dividing by 100, i.e. the last two digits: 20170315 gives 15.
6. **Minutes:** `total_mins = total_secs / 60`, easier to read than seconds.

`.copy()` makes a clean new table after filtering, so adding columns later doesn't trigger pandas warnings.

In [ ]:
before = len(logs)
logs = logs.drop_duplicates()
print("Duplicate rows removed:", before - len(logs))

before = len(logs)
logs = logs[(logs[num_cols] >= 0).all(axis=1)].copy()
print("Rows with negative or missing values removed:", before - len(logs))

too_long = (logs["total_secs"] > 86400).sum()
logs["total_secs"] = logs["total_secs"].clip(upper=86400)
print("Rows with more than 24 hours of listening clipped to 24 hours:", too_long)

logs = logs[(logs["date"] >= 20170301) & (logs["date"] <= 20170331)].copy()
logs["day"] = logs["date"] % 100
logs["total_mins"] = logs["total_secs"] / 60
print("Date range:", logs["date"].min(), "to", logs["date"].max())

# Step 3: One row per user per day, and a first look

**One row per user per day.**
Some users have several log rows on the same day (for example from different devices), so we **add them up** with `groupby(["msno", "day"])` and `.sum()`. `as_index=False` keeps `msno` and `day` as normal columns.

`active = 1` marks "this user listened on this day". Days without listening have no row at all; they will become 0 in Step 5.
We also print how many rows were merged into an existing user-day.

In [ ]:
feature_cols = ["num_25", "num_50", "num_75", "num_985", "num_100", "num_unq", "total_mins"]

daily = logs.groupby(["msno", "day"], as_index=False)[feature_cols].sum()
daily["active"] = 1
print("Rows merged into an existing user-day:", len(logs) - len(daily))
print(daily.shape)
daily.head()

**EDA: how many users listened on each day?** For every day of March, we count the unique users who listened.

What to look for:
- a **weekly pattern** (more or less listening on some days of the week; 1 March 2017 was a Wednesday, so the weekends are days 4 and 5, 11 and 12, 18 and 19, and 25 and 26);
- a day with **suspiciously few users**, which would suggest a **logging gap**: data missing because of a technical problem, not because users stopped listening. A gap in the prediction window would make users look like churners when they weren't.

In [ ]:
daily.groupby("day")["msno"].nunique().plot(kind="bar", figsize=(12, 4), color="steelblue")
plt.title("Users who listened on each day of March 2017 (sample)")
plt.xlabel("Day of March")
plt.ylabel("Active users")
plt.show()

*Pending: this explanation is written from the real results, after the first run on the KKBox data.* <!-- PLACEHOLDER_DAYS -->

# Step 4: Build the labels

**Build the labels** with the time split:

```
Days 1 to 21     OBSERVATION WINDOW   the model's input
Days 22 to 31    PREDICTION WINDOW    churn = 1 if the user does NOT listen at all here
```

- `obs` = the rows for days 1 to 21 (the input), and `future` = the rows for days 22 to 31 (only used for the label).
- `users` = everyone who listened at least once in days 15 to 21, sorted so the order is always the same. `pd.Index` makes the lookups below fast.
- `users.isin(future["msno"].unique())` is True for users who appear in the prediction window, i.e. who listened again. `~` flips it, so "not seen in days 22 to 31" becomes churn = 1. `.astype(int)` turns True/False into 1/0.

**Sanity check:** the churn rate should be between 2% and 40%. Outside that range, something would be wrong with the data or the windows.

In [ ]:
obs = daily[daily["day"] <= 21]
future = daily[daily["day"] >= 22]

users = pd.Index(sorted(obs.loc[obs["day"] >= 15, "msno"].unique()))
y = (~users.isin(future["msno"].unique())).astype(int)

print(f"Currently active users (listened in days 15 to 21): {len(users):,}")
print(f"Churn rate (no listening in days 22 to 31): {y.mean() * 100:.1f}%")

*Pending: this explanation is written from the real results, after the first run on the KKBox data.* <!-- PLACEHOLDER_LABELS -->

# Step 5: Build the sequences

**Build the sequences: a 3D array.**
- `pivot` turns the long table (one row per user and day) into a **wide table**: one row per user, one column per day. We do this once for every feature (the 7 listening features plus `active`).
- `reindex(index=users, columns=range(1, 22))` makes sure every wide table has **all** the users, in the same order, and **all 21 days**. Days without listening become `NaN` (missing), and `fillna(0)` turns them into 0: no listening.
- `np.stack(arrays, axis=-1)` stacks the 8 wide tables along a new last axis. The result is a **3D array: (users, 21 days, features)**.
  Picture a stack of small spreadsheets, one per user, each with 21 rows (days) and one column per feature.
- We split it into `raw` (the 7 listening features) and `active` (the 0/1 flag). `float32` is the standard number type for neural networks (Lesson 7).

Expected shapes: `(number of users, 21, 7)` and `(number of users, 21)`.

In [ ]:
obs = obs[obs["msno"].isin(users)]

arrays = []
for col in feature_cols + ["active"]:
    wide = obs.pivot(index="msno", columns="day", values=col)
    wide = wide.reindex(index=users, columns=range(1, 22)).fillna(0)
    arrays.append(wide.values)

all_days = np.stack(arrays, axis=-1).astype("float32")
raw = all_days[:, :, :7]       # the 7 listening features
active = all_days[:, :, 7]     # 1 = listened that day, 0 = didn't

print("raw shape:", raw.shape, "| active shape:", active.shape)

**Check our work.** The 3D array should contain exactly the same listening days as the table:
- the number of user-days in the table must equal the number of 1s in `active`;
- every user must have listened at least once in days 15 to 21 (that's how we chose them). In the array, days 15 to 21 are positions 14 to 20, because Python counts from 0.

In [ ]:
print("User-days in the table :", len(obs))
print("Active days in the array:", int(active.sum()))
print("Users with no listening in days 15 to 21:", int((active[:, 14:].sum(axis=1) == 0).sum()))

**EDA: do future churners behave differently?**
For each day from 1 to 21, we compare the users who stayed active with the users who churned (using the labels we just built):
- **left:** the **share of users who listened** that day. The mean of the 0/1 `active` flag is the share of 1s, the same trick as the churn rate in Lesson 3;
- **right:** the **average minutes listened** that day.

`active[y == 0]` selects the users who stayed active, and `.mean(axis=0)` averages over those users, giving one value per day.

In [ ]:
days = range(1, 22)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(days, active[y == 0].mean(axis=0), label="Stayed active")
axes[0].plot(days, active[y == 1].mean(axis=0), label="Churned (silent in days 22 to 31)")
axes[0].set_title("Share of users who listened each day")
axes[0].set_xlabel("Day of March")
axes[0].legend()

mins = raw[:, :, feature_cols.index("total_mins")]
axes[1].plot(days, mins[y == 0].mean(axis=0), label="Stayed active")
axes[1].plot(days, mins[y == 1].mean(axis=0), label="Churned (silent in days 22 to 31)")
axes[1].set_title("Average minutes listened per day")
axes[1].set_xlabel("Day of March")
axes[1].legend()

plt.show()

*Pending: this explanation is written from the real results, after the first run on the KKBox data.* <!-- PLACEHOLDER_EDA -->

# Step 6: Save

**Save the sequences** in one compressed NumPy file, so notebook 07 doesn't need to read the big CSV again.
- `np.savez_compressed` stores several arrays in one `.npz` file: `raw`, `active`, `y` and the user ids.
- `np.array(users, dtype=str)` stores the user ids as plain text, so the file loads safely on any pandas version.
- The file goes into `data/kkbox/`, which is ignored by git.

In [ ]:
np.savez_compressed(f"{DATA_DIR}/sequences.npz",
                    raw=raw, active=active, y=y, users=np.array(users, dtype=str))
print("Saved", f"{DATA_DIR}/sequences.npz")

*Pending: this explanation is written from the real results, after the first run on the KKBox data.* <!-- PLACEHOLDER_SUMMARY06 -->